# DistilBERT for GitHub issue triage (issue type and tags)

**How to use**
1. Runtime > Change runtime type > **T4 GPU**.
2. Run the cells from top to bottom.
3. Keep `SMOKE = True` the first time. It trains on a few hundred issues for 1 epoch and takes about 2 to 3 minutes, to check that everything works.
4. Then set `SMOKE = False` in the config cell and run again from the config cell down.

Everything is saved to your Google Drive folder `issue-triage/runs`, and training resumes automatically if Colab disconnects (just re-run the cells).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SMOKE = True            # True = quick test on a small sample. Set to False for the real run.

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256           # tokens per issue (title + start of the body)
BATCH = 32
EPOCHS = 3
LR = 3e-5
SEED = 42
RUN_SUFFIX = ""         # for a new experiment use e.g. "-len512" so earlier results are not overwritten
DRIVE = "/content/drive/MyDrive/issue-triage"

TYPES = ["bug", "docs", "feature", "question"]      # same order as the scikit-learn baselines
TAGS = ["api", "build_ci", "needs_discussion", "needs_info", "performance", "regression", "spam_invalid"]

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

OUT = f"{DRIVE}/runs"
os.makedirs(OUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU: Runtime > Change runtime type > T4 GPU")

In [ ]:
df = pd.read_parquet(f"{DRIVE}/issues_clean.parquet")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def get_data(task):
    """Same subsets and time-based splits as the scikit-learn baselines."""
    d = df.dropna(subset=["issue_type"]) if task == "type" else df[df["labels"].apply(len) > 0]
    parts = {}
    for s in ["train", "val", "test"]:
        x = d[d["split"] == s]
        if SMOKE:
            x = x.sample(min(len(x), 400 if s == "train" else 200), random_state=SEED)
        ids = tok(list(x["text_clean"]), truncation=True, max_length=MAX_LEN)["input_ids"]
        if task == "type":
            y = x["issue_type"].map({t: i for i, t in enumerate(TYPES)}).values.astype("int64")
        else:
            y = np.array([[t in set(tags) for t in TAGS] for tags in x["tags"]], dtype="float32")
        parts[s] = (ids, y)
        print(f"{task} {s}: {len(ids)} issues")
    return parts

def make_loader(ids, y, shuffle):
    items = [{"input_ids": a, "labels": b} for a, b in zip(ids, y)]
    def collate(batch):
        enc = tok.pad([{"input_ids": b["input_ids"]} for b in batch], return_tensors="pt")
        enc["labels"] = torch.tensor(np.stack([b["labels"] for b in batch]))
        return enc
    return DataLoader(items, batch_size=BATCH if shuffle else BATCH * 2, shuffle=shuffle, collate_fn=collate)

In [ ]:
def type_metrics(logits, y):
    p = logits.argmax(1)
    m = {"macro_f1": float(f1_score(y, p, average="macro", labels=list(range(len(TYPES))), zero_division=0)),
         "accuracy": float(accuracy_score(y, p))}
    for i, t in enumerate(TYPES):
        m[f"f1_{t}"] = float(f1_score(y == i, p == i, zero_division=0))
    return m

def tag_metrics(logits, y):
    p = (1 / (1 + np.exp(-logits))) > 0.5          # default threshold, as in the baselines
    m = {"micro_f1": float(f1_score(y, p, average="micro", zero_division=0)),
         "macro_f1": float(f1_score(y, p, average="macro", zero_division=0))}
    for j, t in enumerate(TAGS):
        m[f"f1_{t}"] = float(f1_score(y[:, j], p[:, j], zero_division=0))
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval()
    outs = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            outs.append(model(**batch).logits.float().cpu())
    return torch.cat(outs).numpy()

def train(task):
    data = get_data(task)
    n_labels = len(TYPES) if task == "type" else len(TAGS)
    metrics_fn = type_metrics if task == "type" else tag_metrics
    run_dir = f"{OUT}/{task}{RUN_SUFFIX}{'-smoke' if SMOKE else ''}"
    os.makedirs(run_dir, exist_ok=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_labels).to(device)
    y_tr = data["train"][1]
    if task == "type":      # balanced class weights, like class_weight="balanced" in the baselines
        counts = np.bincount(y_tr, minlength=n_labels)
        w = torch.tensor(len(y_tr) / (n_labels * np.maximum(counts, 1)), dtype=torch.float32, device=device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=w)
    else:                   # positive weight per tag, capped at 10
        pos = y_tr.sum(0)
        pw = torch.tensor(np.clip((len(y_tr) - pos) / np.maximum(pos, 1), 1, 10), dtype=torch.float32, device=device)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pw)

    train_loader = make_loader(*data["train"], True)
    val_loader = make_loader(*data["val"], False)
    test_loader = make_loader(*data["test"], False)

    steps = EPOCHS * len(train_loader)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    last_path = f"{run_dir}/last.pt"
    start_epoch, best, history = 0, -1.0, []
    if os.path.exists(last_path):                                  # resume after a disconnect
        ck = torch.load(last_path, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        start_epoch, best, history = ck["epoch"], ck["best"], ck["history"]
        print(f"resumed after epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        t0, total = time.time(), 0.0
        for batch in train_loader:
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels if task == "type" else labels.float())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            total += loss.item()

        m = metrics_fn(predict(model, val_loader), data["val"][1])
        history.append({"epoch": epoch + 1, "train_loss": total / len(train_loader), **{f"val_{k}": v for k, v in m.items()}})
        print(f"epoch {epoch+1}/{EPOCHS} | {time.time()-t0:.0f}s | train loss {total/len(train_loader):.4f} | val macro-F1 {m['macro_f1']:.4f}")
        if m["macro_f1"] > best:                                    # keep the best epoch
            best = m["macro_f1"]
            model.save_pretrained(f"{run_dir}/best"); tok.save_pretrained(f"{run_dir}/best")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "epoch": epoch + 1, "best": best, "history": history}, last_path)

    model = AutoModelForSequenceClassification.from_pretrained(f"{run_dir}/best").to(device)
    res, logits = {}, {}
    for split, loader in [("val", val_loader), ("test", test_loader)]:
        logits[split] = predict(model, loader)
        np.save(f"{run_dir}/{split}_logits.npy", logits[split])
        res[split] = metrics_fn(logits[split], data[split][1])
    if task == "type":
        y = np.concatenate([data["val"][1], data["test"][1]])
        p = np.concatenate([logits["val"], logits["test"]]).argmax(1)
        res["valtest_f1_question"] = float(f1_score(y == 3, p == 3, zero_division=0))
        print(classification_report(data["test"][1], logits["test"].argmax(1), labels=list(range(4)),
                                    target_names=TYPES, zero_division=0))
    res["history"] = history
    json.dump(res, open(f"{run_dir}/results.json", "w"), indent=1)
    print("saved to", run_dir)
    return res

## Run: issue type

In [ ]:
res_type = train("type")

## Run: tags (multi-label)

In [ ]:
res_tags = train("tags")

## Compare with the TF-IDF baselines (numbers from MLflow)
Only meaningful when `SMOKE = False`.

In [ ]:
print(f"SMOKE = {SMOKE}   RUN_SUFFIX = '{RUN_SUFFIX}'   (comparison is only meaningful with SMOKE = False)\n")
rows = []
if "res_type" in globals():
    rows += [("type  val macro-F1",            0.863, res_type["val"]["macro_f1"]),
             ("type  test macro-F1",           0.820, res_type["test"]["macro_f1"]),
             ("type  question F1 (val+test)",  0.628, res_type["valtest_f1_question"])]
if "res_tags" in globals():
    rows += [("tags  val micro-F1",            0.359, res_tags["val"]["micro_f1"]),
             ("tags  test micro-F1",           0.400, res_tags["test"]["micro_f1"]),
             ("tags  test macro-F1",           0.430, res_tags["test"]["macro_f1"])]
print(f"{'':32s}{'TF-IDF baseline':>16s}{'DistilBERT':>12s}")
for name, base, new in rows:
    print(f"{name:32s}{base:16.3f}{new:12.3f}")
if "res_type" in globals():
    print("\ntype per-class F1 (test):", {t: round(res_type['test'][f'f1_{t}'], 3) for t in TYPES})
    print("type per-class F1 (val): ", {t: round(res_type['val'][f'f1_{t}'], 3) for t in TYPES})
    for h in res_type["history"]:
        print({k: round(v, 3) for k, v in h.items()})
if "res_tags" in globals():
    print("\nper-tag test F1:", {t: round(res_tags['test'][f'f1_{t}'], 3) for t in TAGS})

## Later: upload the best models to the Hugging Face Hub (needed for deployment)
Not needed now. We will do this after the full run. It uses a token stored in Colab Secrets (key icon on the left), never pasted into a cell.

In [ ]:
# from huggingface_hub import login
# from google.colab import userdata
# login(userdata.get("HF_TOKEN"))
# HF_USER = "YOUR_HF_USERNAME"
# for task in ["type", "tags"]:
#     m = AutoModelForSequenceClassification.from_pretrained(f"{OUT}/{task}/best")
#     m.push_to_hub(f"{HF_USER}/issue-triage-distilbert-{task}")
#     tok.push_to_hub(f"{HF_USER}/issue-triage-distilbert-{task}")